# Sales Forecasting Dashboard

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eddmpython/vectrix/blob/master/notebooks/showcase/01_sales_forecasting_dashboard.ipynb)

**A complete demand forecasting workflow you can drop into your business today.**

This notebook demonstrates a production-ready forecasting pipeline with interactive Plotly visualizations. Every chart is publication-quality and tells a clear business story.

**What you'll build:**
1. Interactive forecast chart with confidence intervals
2. Model comparison heatmap
3. DNA profiling radar chart
4. What-if scenario comparison
5. Backtest performance over time
6. Business metrics scorecard

In [ ]:
!pip install -q vectrix plotly

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from vectrix import forecast, analyze, compare, loadSample, ForecastDNA
from vectrix.business import AnomalyDetector, WhatIfAnalyzer, Backtester, BusinessMetrics
from vectrix.engine.ets import AutoETS

In [ ]:
COLORS = {
    "primary": "#6366f1",
    "accent": "#a855f7",
    "positive": "#22c55e",
    "negative": "#ef4444",
    "warning": "#f59e0b",
    "muted": "#94a3b8",
    "bg": "#0f172a",
    "card": "#1e293b",
    "text": "#f1f5f9",
}

LAYOUT = dict(
    template="plotly_dark",
    paper_bgcolor=COLORS["bg"],
    plot_bgcolor=COLORS["bg"],
    font=dict(family="Inter, sans-serif", color=COLORS["text"]),
    margin=dict(l=60, r=30, t=60, b=40),
)

---

## Step 1: Load & Analyze Data

In [ ]:
df = loadSample("retail")
print(f"Dataset: {len(df)} rows, {df.columns.tolist()}")
print(f"Date range: {df['date'].iloc[0]} to {df['date'].iloc[-1]}")
df.head()

In [ ]:
report = analyze(df, date="date", value="sales")
print(report.summary())

---

## Step 2: Interactive Forecast Chart

In [ ]:
result = forecast(df, date="date", value="sales", steps=30)
print(f"Model: {result.model} | MAPE: {result.mape:.2f}% | RMSE: {result.rmse:.2f}")

In [ ]:
forecast_df = result.toDataframe()
forecast_dates = pd.to_datetime(forecast_df["date"])
hist_dates = pd.to_datetime(df["date"])

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=hist_dates, y=df["sales"],
    name="Historical", line=dict(color=COLORS["muted"], width=1.5),
    hovertemplate="%{x|%Y-%m-%d}<br>Sales: %{y:,.0f}<extra></extra>",
))

fig.add_trace(go.Scatter(
    x=forecast_dates, y=result.upper,
    name="Upper 95%", line=dict(width=0), showlegend=False,
    hoverinfo="skip",
))
fig.add_trace(go.Scatter(
    x=forecast_dates, y=result.lower,
    name="95% CI", fill="tonexty",
    fillcolor=f"rgba(99,102,241,0.15)",
    line=dict(width=0),
    hoverinfo="skip",
))

fig.add_trace(go.Scatter(
    x=forecast_dates, y=result.predictions,
    name=f"Forecast ({result.model})",
    line=dict(color=COLORS["primary"], width=2.5, dash="dot"),
    hovertemplate="%{x|%Y-%m-%d}<br>Forecast: %{y:,.0f}<extra></extra>",
))

anomaly_idx = report.anomalies
if len(anomaly_idx) > 0:
    fig.add_trace(go.Scatter(
        x=hist_dates[anomaly_idx], y=df["sales"].values[anomaly_idx],
        name="Anomalies", mode="markers",
        marker=dict(color=COLORS["negative"], size=8, symbol="x"),
        hovertemplate="Anomaly at %{x|%Y-%m-%d}<br>Value: %{y:,.0f}<extra></extra>",
    ))

fig.update_layout(
    **LAYOUT,
    title=dict(text=f"Sales Forecast — {result.model} (MAPE {result.mape:.1f}%)", font_size=18),
    xaxis_title="Date", yaxis_title="Sales",
    legend=dict(orientation="h", y=-0.15),
    height=500,
)
fig.show()

---

## Step 3: Model Comparison Heatmap

In [ ]:
ranking = compare(df, date="date", value="sales", steps=30)
top10 = ranking.head(10).copy()

metrics_cols = ["mape", "rmse", "mae", "smape"]
normalized = top10[metrics_cols].copy()
for col in metrics_cols:
    mn, mx = normalized[col].min(), normalized[col].max()
    if mx > mn:
        normalized[col] = (normalized[col] - mn) / (mx - mn)
    else:
        normalized[col] = 0

fig = go.Figure(data=go.Heatmap(
    z=normalized[metrics_cols].values,
    x=[c.upper() for c in metrics_cols],
    y=top10["model"].values,
    colorscale=[[0, COLORS["positive"]], [0.5, COLORS["warning"]], [1, COLORS["negative"]]],
    text=top10[metrics_cols].round(2).values,
    texttemplate="%{text}",
    textfont=dict(size=12),
    showscale=False,
    hovertemplate="%{y}<br>%{x}: %{text}<extra></extra>",
))

fig.update_layout(
    **LAYOUT,
    title=dict(text="Top 10 Models — Error Heatmap", font_size=18),
    height=400,
    yaxis=dict(autorange="reversed"),
)
fig.show()

---

## Step 4: DNA Profile Radar Chart

In [ ]:
dna = report.dna
feat = dna.features

radar_keys = ["trendStrength", "seasonalStrength", "hurstExponent",
              "volatilityClustering", "nonlinearAutocorr", "forecastability"]
radar_labels = ["Trend", "Seasonality", "Memory (Hurst)",
                "Vol. Clustering", "Nonlinear", "Forecastability"]

radar_values = []
for k in radar_keys:
    v = feat.get(k, 0)
    if v is None:
        v = 0
    radar_values.append(min(float(v), 1.0))
radar_values.append(radar_values[0])
radar_labels_closed = radar_labels + [radar_labels[0]]

fig = go.Figure()
fig.add_trace(go.Scatterpolar(
    r=radar_values, theta=radar_labels_closed,
    fill="toself", fillcolor=f"rgba(99,102,241,0.2)",
    line=dict(color=COLORS["primary"], width=2),
    name="DNA Profile",
))

fig.update_layout(
    **LAYOUT,
    title=dict(text=f"DNA Profile — {dna.category} ({dna.difficulty}, score {dna.difficultyScore:.0f}/100)", font_size=18),
    polar=dict(
        bgcolor=COLORS["card"],
        radialaxis=dict(visible=True, range=[0, 1], gridcolor="rgba(255,255,255,0.1)"),
        angularaxis=dict(gridcolor="rgba(255,255,255,0.1)"),
    ),
    height=450,
)
fig.show()

---

## Step 5: What-If Scenario Comparison

In [ ]:
analyzer = WhatIfAnalyzer()
scenarios = analyzer.analyze(
    result.predictions,
    df["sales"].values,
    [
        {"name": "Base Case", "trend_change": 0.0},
        {"name": "Growth +15%", "trend_change": 0.15},
        {"name": "Recession -20%", "trend_change": -0.20},
        {"name": "Supply Shock", "shock_at": 10, "shock_magnitude": -0.3, "shock_duration": 7},
    ]
)

scenario_colors = [COLORS["primary"], COLORS["positive"], COLORS["negative"], COLORS["warning"]]

fig = go.Figure()
for i, s in enumerate(scenarios):
    fig.add_trace(go.Scatter(
        x=list(range(1, len(s.predictions) + 1)),
        y=s.predictions,
        name=f"{s.name} ({s.impact:+.1f}%)",
        line=dict(color=scenario_colors[i % len(scenario_colors)], width=2),
        hovertemplate=f"{s.name}<br>Step %{{x}}<br>Value: %{{y:,.0f}}<extra></extra>",
    ))

fig.update_layout(
    **LAYOUT,
    title=dict(text="What-If Scenario Analysis", font_size=18),
    xaxis_title="Forecast Step", yaxis_title="Predicted Value",
    legend=dict(orientation="h", y=-0.15),
    height=450,
)
fig.show()

---

## Step 6: Backtest Performance

In [ ]:
bt = Backtester(nFolds=5, horizon=14, strategy='expanding')
bt_result = bt.run(df["sales"].values, lambda: AutoETS())

fold_mapes = [f.mape for f in bt_result.folds]
fold_rmses = [f.rmse for f in bt_result.folds]
fold_nums = [f"Fold {f.fold}" for f in bt_result.folds]

fig = make_subplots(rows=1, cols=2, subplot_titles=("MAPE by Fold", "RMSE by Fold"))

bar_colors = [COLORS["positive"] if m == min(fold_mapes) else
              COLORS["negative"] if m == max(fold_mapes) else
              COLORS["primary"] for m in fold_mapes]

fig.add_trace(go.Bar(
    x=fold_nums, y=fold_mapes, marker_color=bar_colors,
    text=[f"{m:.1f}%" for m in fold_mapes], textposition="auto",
    name="MAPE", showlegend=False,
), row=1, col=1)

fig.add_trace(go.Bar(
    x=fold_nums, y=fold_rmses, marker_color=COLORS["accent"],
    text=[f"{r:.1f}" for r in fold_rmses], textposition="auto",
    name="RMSE", showlegend=False,
), row=1, col=2)

fig.add_hline(y=bt_result.avgMAPE, line_dash="dash", line_color=COLORS["warning"],
              annotation_text=f"Avg {bt_result.avgMAPE:.1f}%", row=1, col=1)
fig.add_hline(y=bt_result.avgRMSE, line_dash="dash", line_color=COLORS["warning"],
              annotation_text=f"Avg {bt_result.avgRMSE:.1f}", row=1, col=2)

fig.update_layout(
    **LAYOUT,
    title=dict(text=f"Backtest Results — {bt_result.avgMAPE:.1f}% Average MAPE", font_size=18),
    height=400,
)
fig.show()

---

## Step 7: Business Metrics Scorecard

In [ ]:
np.random.seed(42)
n = 30
actual = df["sales"].values[-n:]
noise = np.random.randn(n) * actual.std() * 0.05
predicted = actual + noise

bm = BusinessMetrics()
metrics = bm.calculate(actual, predicted)

metric_names = ["Accuracy", "Bias %", "WAPE", "MASE"]
metric_values = [
    metrics["forecastAccuracy"],
    metrics["biasPercent"],
    metrics["wape"],
    metrics["mase"],
]
metric_thresholds = [95, 0, 5, 1.0]
metric_formats = ["{:.1f}%", "{:+.2f}%", "{:.2f}%", "{:.3f}"]

def score_color(name, val, threshold):
    if name == "Accuracy":
        return COLORS["positive"] if val >= threshold else COLORS["negative"]
    elif name == "Bias %":
        return COLORS["positive"] if abs(val) < 3 else COLORS["warning"] if abs(val) < 5 else COLORS["negative"]
    elif name == "WAPE":
        return COLORS["positive"] if val < threshold else COLORS["warning"] if val < 10 else COLORS["negative"]
    else:
        return COLORS["positive"] if val < threshold else COLORS["negative"]

fig = go.Figure()
for i, (name, val, thresh, fmt) in enumerate(zip(metric_names, metric_values, metric_thresholds, metric_formats)):
    color = score_color(name, val, thresh)
    fig.add_trace(go.Indicator(
        mode="number",
        value=val,
        number=dict(font=dict(size=40, color=color), valueformat=fmt.replace("{", "").replace("}", "").replace(":", "")),
        title=dict(text=name, font=dict(size=16, color=COLORS["text"])),
        domain=dict(row=0, column=i),
    ))

fig.update_layout(
    **LAYOUT,
    grid=dict(rows=1, columns=4, pattern="independent"),
    title=dict(text="Business Metrics Scorecard", font_size=18),
    height=250,
)
fig.show()

---

## Summary

This notebook demonstrated a complete forecasting pipeline that you can adapt for your business:

1. **Load data** — any CSV, DataFrame, or built-in sample
2. **Analyze** — DNA profiling reveals data characteristics before forecasting
3. **Forecast** — automatic model selection from 30+ candidates
4. **Compare** — see how all models rank against each other
5. **Scenario plan** — what-if analysis for budget/risk planning
6. **Validate** — backtesting proves the approach works on your data
7. **Measure** — business metrics that stakeholders understand

**To use with your own data:**
```python
df = pd.read_csv("your_data.csv")
result = forecast(df, date="your_date_col", value="your_value_col", steps=30)
```

---

**Resources:**
- [Vectrix Documentation](https://eddmpython.github.io/vectrix/docs/)
- [GitHub](https://github.com/eddmpython/vectrix)
- [PyPI](https://pypi.org/project/vectrix/)